# Multi-Agent Architecture Demo

**LegalMind-AI | Azure AI Engineer Portfolio**

This notebook demonstrates the multi-agent orchestration system that powers LegalMind-AI.
We walk through the complete pipeline from intent classification to agent execution,
including handoffs between specialist agents and responsible AI guardrails.

### Architecture

```
User Message
    |
    v
[Input Guardrails]  -- content safety, PII detection
    |
    v
[Orchestrator]  -- GPT-4o intent classification (temp=0.1)
    |
    +-- document_analysis  --> DocumentAgent (Semantic Kernel + tools)
    +-- legal_research     --> ResearchAgent (search + summarize tools)
    +-- drafting           --> DraftingAgent (draft + translate tools)
    +-- compliance         --> ComplianceAgent (search + extract tools)
    +-- general            --> Orchestrator handles directly
    |
    v
[Output Guardrails]  -- content safety, PII, groundedness
    |
    v
Response to User
```

Each specialist agent extends `BaseAgent` and uses **Semantic Kernel** for chat completion
with automatic tool calling. Tools are registered as SK plugins.

> **Note**: All service calls are simulated. In production, these connect to Azure OpenAI,
> Azure AI Search, and Azure Content Safety via managed identity authentication.

## 1. Setup

In [ ]:
import json
import sys
import time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Import orchestrator types (data models only -- no live service connections)
from legalmind.agents.orchestrator import Intent, RoutingDecision, OrchestratorResponse
from legalmind.agents.base_agent import AgentResponse
from legalmind.agents.prompts.system_prompts import ORCHESTRATOR_PROMPT
from legalmind.context.guardrails import GuardrailVerdict, GroundednessResult, GuardrailResult

print(f"Project root: {PROJECT_ROOT}")
print(f"Available intents: {[i.value for i in Intent]}")

## 2. Intent Classification

The Orchestrator classifies user intent using GPT-4o with a specialized system prompt.
It returns a JSON response with the intent, confidence, reasoning, and detected language.

```python
# Production code from orchestrator.py:
orchestrator = Orchestrator(classification_temperature=0.1)
routing = await orchestrator.classify_intent(user_message)
print(routing.intent, routing.confidence, routing.reasoning)
```

Let's test classification with different query types.

In [ ]:
# Simulate intent classification for various query types
# In production, this calls GPT-4o with the ORCHESTRATOR_PROMPT

test_queries = [
    # Document analysis queries
    ("Kun je dit contract samenvatten en de risico's identificeren?",
     Intent.DOCUMENT_ANALYSIS, 0.95, "nl",
     "User wants to analyze and summarize a contract with risk assessment."),
    
    ("Compare these two versions of the insurance policy.",
     Intent.DOCUMENT_ANALYSIS, 0.92, "en",
     "User requests document comparison of policy versions."),
    
    # Legal research queries
    ("Wat zegt de wet over werkgeversaansprakelijkheid bij arbeidsongevallen?",
     Intent.LEGAL_RESEARCH, 0.94, "nl",
     "User asks about statutory provisions regarding employer liability."),
    
    ("Zoek relevante jurisprudentie over afspiegelingsbeginsel bij reorganisatie.",
     Intent.LEGAL_RESEARCH, 0.97, "nl",
     "User requests case law research on specific legal principle."),
    
    # Drafting queries
    ("Schrijf een sommatiebrief voor het terugvorderen van de waarborgsom.",
     Intent.DRAFTING, 0.96, "nl",
     "User requests drafting a demand letter in Dutch."),
    
    # Compliance queries
    ("Is ons privacybeleid compliant met de AVG?",
     Intent.COMPLIANCE, 0.93, "nl",
     "User asks for GDPR/AVG compliance check."),
    
    # General / ambiguous queries
    ("Hallo, wat kun je voor mij doen?",
     Intent.GENERAL, 0.88, "nl",
     "Greeting and capability inquiry."),
    
    ("What is the weather like today?",
     Intent.GENERAL, 0.35, "en",
     "Non-legal query; outside domain scope."),
]

print("=== Intent Classification Results ===")
print(f"{'Query':<65} {'Intent':<22} {'Conf':>5} {'Lang':>4}")
print("=" * 100)

for query, intent, confidence, lang, reasoning in test_queries:
    # Truncate query for display
    display_query = query[:62] + "..." if len(query) > 65 else query
    print(f"{display_query:<65} {intent.value:<22} {confidence:>5.2f} {lang:>4}")

print(f"\nTotal queries classified: {len(test_queries)}")
print(f"Average confidence: {sum(c for _, _, c, _, _ in test_queries) / len(test_queries):.2f}")

In [ ]:
# Show the orchestrator prompt structure
print("=== Orchestrator System Prompt (excerpt) ===")
lines = ORCHESTRATOR_PROMPT.split("\n")
for line in lines[:15]:
    print(f"  {line}")
print(f"  ...")
print(f"  [{len(lines) - 15} more lines]")
print(f"\n  Total prompt length: {len(ORCHESTRATOR_PROMPT)} chars ({len(ORCHESTRATOR_PROMPT.split())} words)")
print(f"  Classification temperature: 0.1 (highly deterministic for routing)")

## 3. End-to-End Document Analysis Query

Let's trace a complete request through the pipeline, from user query to final response.

In [ ]:
# Simulate an end-to-end document analysis request
user_message = "Analyseer deze verzekeringspolis en geef een samenvatting met de belangrijkste dekkingsgebieden."

print("=" * 80)
print("END-TO-END DOCUMENT ANALYSIS REQUEST")
print("=" * 80)

# Step 1: Input guardrails
print("\n[Step 1] Input Guardrails")
print(f"  Content safety: PASS (no harmful content detected)")
print(f"  PII detection: PASS (no PII in query)")
print(f"  Verdict: PASS")

# Step 2: Intent classification
routing = RoutingDecision(
    intent=Intent.DOCUMENT_ANALYSIS,
    confidence=0.96,
    reasoning="User requests document analysis and summary of an insurance policy.",
    secondary_intents=[],
    requires_context=False,
    language_detected="nl",
)

print(f"\n[Step 2] Intent Classification")
print(f"  Intent: {routing.intent.value}")
print(f"  Confidence: {routing.confidence}")
print(f"  Reasoning: {routing.reasoning}")
print(f"  Language: {routing.language_detected}")

# Step 3: Agent invocation
print(f"\n[Step 3] Agent Invocation: DocumentAgent")
print(f"  System prompt: DOCUMENT_AGENT_PROMPT (750 words)")
print(f"  Few-shot examples: 4 messages (2 pairs)")
print(f"  Temperature: 0.3")
print(f"  Tool calling: ENABLED (extract_tool, search_tool)")

# Step 4: Tool calls
print(f"\n[Step 4] Tool Execution")
print(f"  Tool 1: extract_tool.extract_entities")
print(f"    Input: policy text (3,200 chars)")
print(f"    Output: 12 entities (Organization, DateTime, Quantity, etc.)")
print(f"  Tool 2: extract_tool.extract_key_phrases")
print(f"    Input: policy text (3,200 chars)")
print(f"    Output: 18 key phrases")

# Step 5: Agent response
simulated_response = """\
## Samenvatting Verzekeringspolis

**Documenttype**: Rechtsbijstandverzekering
**Verzekeraar**: Voorbeeld Rechtsbijstand N.V.
**Polisnummer**: VP-2024-001234
**Looptijd**: 1 januari 2024 - 31 december 2024

### Dekkingsgebieden

| Rechtsgebied | Dekking | Wachttijd |
|-------------|---------|----------|
| Arbeidsrecht | Gedekt | 3 maanden |
| Consumentenrecht | Gedekt | Geen |
| Huurrecht | Gedekt | 3 maanden |
| Verkeersrecht | Gedekt | Geen |
| Burenrecht | Gedekt | Geen |

### Financiele Gegevens
- **Maximale dekking**: EUR 50.000 per geschil (Art. 2.2)
- **Eigen risico**: EUR 250 per geschil (Art. 4)
- **Jaarpremie**: EUR 180,00 incl. assurantiebelasting (Art. 6.1)

### Uitsluitingen (Art. 3)
- Pre-existente geschillen
- Geschillen met de verzekeraar zelf
- Strafzaken
- Onroerend goed geschillen
- Fiscale geschillen

### Aandachtspunten
- Vrije advocaatkeuze bij gerechtelijke procedures (Art. 10)
- Geschillenregeling via Kifid beschikbaar (Art. 11)
- Stilzwijgende verlenging na eerste jaar (Art. 7)"""

agent_response = AgentResponse(
    content=simulated_response,
    agent_name="document_analysis",
    tool_calls=["extract_tool-extract_entities", "extract_tool-extract_key_phrases"],
    metadata={
        "elapsed_seconds": 3.42,
        "history_length": 8,
        "temperature": 0.3,
    },
    success=True,
)

print(f"\n[Step 5] Agent Response")
print(f"  Agent: {agent_response.agent_name}")
print(f"  Tools used: {agent_response.tool_calls}")
print(f"  Response length: {len(agent_response.content)} chars")
print(f"  Elapsed: {agent_response.metadata['elapsed_seconds']}s")

# Step 6: Output guardrails
print(f"\n[Step 6] Output Guardrails")
print(f"  Content safety: PASS")
print(f"  PII detection: PASS (no client PII in output)")
print(f"  Groundedness: 0.94 (PASS, threshold=0.6)")
print(f"  Final verdict: PASS -- delivering response")

print(f"\n{'=' * 80}")
print(f"FINAL RESPONSE:")
print(f"{'=' * 80}")
print(agent_response.content)

## 4. Agent Handoff Demo

When one agent determines that another specialist is better suited for part of the
request, it signals a handoff via `metadata["handoff_to"]`. The Orchestrator then
routes to the target agent with context from the first agent's analysis.

```python
# From orchestrator.py:
if self._enable_handoffs and self._should_handoff(agent_response):
    agent_response, handoff_chain = await self._execute_handoff(
        original_response=agent_response,
        user_message=user_message,
        context=context,
        handoff_chain=handoff_chain,
    )
```

In [ ]:
# Demonstrate a Research -> Drafting handoff scenario
user_message_handoff = (
    "Onderzoek de juridische gronden voor werkgeversaansprakelijkheid bij ons "
    "arbeidsongeval-dossier en stel vervolgens een sommatiebrief op."
)

print("=== Agent Handoff Demo ===")
print(f"\nUser: '{user_message_handoff}'\n")

# Step 1: Orchestrator routes to Research Agent (dominant intent)
print("[1] Orchestrator classifies:")
print(f"    Primary intent: legal_research (confidence: 0.78)")
print(f"    Secondary intent: drafting")
print(f"    Reasoning: 'Complex request with research and drafting components. Research is primary.'")

# Step 2: Research Agent processes and signals handoff
research_response = AgentResponse(
    content="""\
## Juridisch Kader: Werkgeversaansprakelijkheid bij Arbeidsongeval

### Toepasselijke wetgeving
**Art. 7:658 BW** -- De werkgever is verplicht de lokalen, werktuigen en gereedschappen 
waarin of waarmee hij de arbeid doet verrichten, op zodanige wijze in te richten en te 
onderhouden als redelijkerwijs nodig is om te voorkomen dat de werknemer schade lijdt.

### Bewijslastverdeling
De werknemer dient te bewijzen dat hij schade heeft geleden in de uitoefening van zijn 
werkzaamheden. De werkgever draagt vervolgens de bewijslast dat hij aan zijn zorgplicht 
heeft voldaan (art. 7:658 lid 2 BW).

### Relevante jurisprudentie
- HR 11 november 2011, ECLI:NL:HR:2011:BR5215
- Rb. Rotterdam 14 mei 2020, ECLI:NL:RBROT:2020:4312

### Conclusie
De juridische grondslag voor aansprakelijkheid is sterk gezien het ontbreken van 
valbeveiliging en actuele RI&E.

---
*Handoff: op basis van deze analyse kan een sommatiebrief worden opgesteld.*""",
    agent_name="legal_research",
    tool_calls=["search_tool-legal_search", "summarize_tool-summarize"],
    metadata={
        "elapsed_seconds": 4.8,
        "handoff_to": "drafting",  # This triggers the handoff
        "handoff_reason": "Research complete. User also requested a demand letter.",
    },
    success=True,
)

print(f"\n[2] ResearchAgent processes:")
print(f"    Tools used: {research_response.tool_calls}")
print(f"    Elapsed: {research_response.metadata['elapsed_seconds']}s")
print(f"    Handoff signal: '{research_response.metadata['handoff_to']}'")
print(f"    Reason: '{research_response.metadata['handoff_reason']}'")

# Step 3: Orchestrator executes handoff to Drafting Agent
print(f"\n[3] Orchestrator executes handoff: legal_research -> drafting")
print(f"    Context passed: prior_analysis (from ResearchAgent)")
print(f"    Handoff depth: 1 (max: 3)")

# Step 4: Drafting Agent generates the letter
drafting_response = AgentResponse(
    content="""\
## Concept Sommatiebrief -- Werkgeversaansprakelijkheid Arbeidsongeval

> Dit is een concept dat professionele beoordeling vereist voor verzending.

[NAAM CLIENT]                              Voorbeeld Rechtsbijstand N.V.
[ADRES]                                    Postbus 12345
                                           1100 AB Amsterdam

Betreft: Aansprakelijkheidsstelling arbeidsongeval d.d. 15 maart 2024
Ons kenmerk: VRB/2024/AO-5678

Geachte heer/mevrouw,

**1.** Namens onze client, de heer J. de Vries, stellen wij u hierbij aansprakelijk 
voor de schade die onze client heeft geleden als gevolg van het arbeidsongeval op 
15 maart 2024.

**2.** Op grond van artikel 7:658 BW is de werkgever verplicht te zorgen voor een 
veilige werkomgeving. Uit onderzoek is gebleken dat de steiger niet was voorzien van 
adequate valbeveiliging (art. 3.16 Arbobesluit).

**3.** De voorlopige schade bedraagt EUR 8.050 (exclusief smartengeld).

**4.** Wij sommeren u om binnen 14 dagen aansprakelijkheid te erkennen.

Hoogachtend,
Voorbeeld Rechtsbijstand N.V.""",
    agent_name="drafting",
    tool_calls=["draft_tool-generate_letter"],
    metadata={"elapsed_seconds": 3.1},
    success=True,
)

print(f"\n[4] DraftingAgent processes:")
print(f"    Tools used: {drafting_response.tool_calls}")
print(f"    Elapsed: {drafting_response.metadata['elapsed_seconds']}s")

# Final result
handoff_chain = ["legal_research", "drafting"]
total_time = research_response.metadata["elapsed_seconds"] + drafting_response.metadata["elapsed_seconds"]

print(f"\n[5] Final Result:")
print(f"    Handoff chain: {' -> '.join(handoff_chain)}")
print(f"    Total elapsed: {total_time:.1f}s")
print(f"    Combined response: research analysis + demand letter")

## 5. Tool Calling in Action

Each specialist agent has access to Semantic Kernel plugins (tools) that extend its
capabilities. Tool calling is automatic -- GPT-4o decides when and how to call tools
based on the conversation context.

```python
# From base_agent.py:
execution_settings.function_choice_behavior = FunctionChoiceBehavior.Auto(
    auto_invoke=True,
    maximum_auto_invoke_attempts=5,
)
```

In [ ]:
# Display the tool registry for each agent
agent_tools = {
    "DocumentAgent": {
        "search_tool": {
            "description": "Search the legal document index using hybrid retrieval",
            "functions": ["legal_search", "search_by_document_type", "search_by_entity"],
        },
        "extract_tool": {
            "description": "Extract structured information from document text",
            "functions": ["extract_entities", "extract_key_phrases", "extract_clauses"],
        },
    },
    "ResearchAgent": {
        "search_tool": {
            "description": "Search case law, statutes, and legal documents",
            "functions": ["legal_search", "search_by_document_type", "search_by_entity"],
        },
        "summarize_tool": {
            "description": "Generate extractive or abstractive summaries",
            "functions": ["summarize", "extract_key_points"],
        },
    },
    "DraftingAgent": {
        "draft_tool": {
            "description": "Generate legal documents from templates",
            "functions": ["generate_letter", "generate_summary", "generate_memo"],
        },
        "translate_tool": {
            "description": "Translate between Dutch and English",
            "functions": ["translate_nl_en", "translate_en_nl"],
        },
    },
    "ComplianceAgent": {
        "search_tool": {
            "description": "Search regulatory requirements and compliance guidelines",
            "functions": ["legal_search", "search_by_document_type", "search_by_entity"],
        },
        "extract_tool": {
            "description": "Extract compliance-relevant information",
            "functions": ["extract_entities", "extract_key_phrases", "extract_clauses"],
        },
    },
}

print("=== Agent Tool Registry ===")
for agent_name, tools in agent_tools.items():
    total_functions = sum(len(t["functions"]) for t in tools.values())
    print(f"\n{agent_name} ({len(tools)} plugins, {total_functions} functions):")
    for tool_name, tool_info in tools.items():
        print(f"  [{tool_name}] {tool_info['description']}")
        for func in tool_info["functions"]:
            print(f"    - {func}()")

In [ ]:
# Simulate a tool-calling trace for a research query
print("=== Tool Calling Trace ===")
print(f"\nUser: 'Zoek jurisprudentie over werkgeversaansprakelijkheid bij arbeidsongevallen'")
print(f"Agent: ResearchAgent")
print()

tool_trace = [
    {
        "step": 1,
        "tool": "search_tool.legal_search",
        "input": {
            "query": "werkgeversaansprakelijkheid arbeidsongeval art. 7:658 BW",
            "document_types": ["case_file"],
            "top": 5,
        },
        "output_summary": "5 results: 3 HR rulings, 2 Rechtbank decisions",
        "latency_ms": 340,
    },
    {
        "step": 2,
        "tool": "search_tool.legal_search",
        "input": {
            "query": "zorgplicht werkgever veilige werkomgeving Arbobesluit",
            "document_types": ["policy"],
            "top": 3,
        },
        "output_summary": "3 results: regulatory guidance on workplace safety",
        "latency_ms": 280,
    },
    {
        "step": 3,
        "tool": "summarize_tool.extract_key_points",
        "input": {
            "texts": ["[5 retrieved case law passages]"],
            "max_points": 5,
        },
        "output_summary": "5 key legal principles extracted",
        "latency_ms": 520,
    },
]

total_latency = 0
for step in tool_trace:
    total_latency += step["latency_ms"]
    print(f"  Step {step['step']}: {step['tool']}")
    print(f"    Input: {json.dumps(step['input'], ensure_ascii=False, indent=None)[:100]}")
    print(f"    Output: {step['output_summary']}")
    print(f"    Latency: {step['latency_ms']}ms")
    print()

print(f"  Total tool execution time: {total_latency}ms")
print(f"  Tool calls: {len(tool_trace)} (within max_auto_invoke_attempts=5)")

## 6. Responsible AI Guardrails Demo

LegalMind implements a three-layer guardrail system from `guardrails.py`:

1. **Input guardrails**: Content safety + PII detection on user messages
2. **Processing guardrails**: Token limits, tool call limits, timeout handling
3. **Output guardrails**: Content safety + PII leak detection + groundedness verification

In [ ]:
# Demonstrate input guardrail scenarios
# In production: result = await guardrails.check_input(user_message)

input_scenarios = [
    {
        "message": "Analyseer dit contract en geef de risico's aan.",
        "verdict": GuardrailVerdict.PASS,
        "content_safe": True,
        "pii_detected": False,
        "warnings": [],
    },
    {
        "message": "Mijn client Jan de Vries, BSN 123456789, is ontslagen.",
        "verdict": GuardrailVerdict.WARN,
        "content_safe": True,
        "pii_detected": True,
        "warnings": ["PII detected (2 entities): Person, NationalID"],
    },
    {
        "message": "[Harmful content example - redacted for notebook]",
        "verdict": GuardrailVerdict.BLOCK,
        "content_safe": False,
        "pii_detected": False,
        "warnings": [],
        "blocked_reason": "Uw bericht bevat inhoud die niet verwerkt kan worden.",
    },
]

print("=== Input Guardrail Scenarios ===")
for i, scenario in enumerate(input_scenarios, 1):
    verdict_symbol = {
        GuardrailVerdict.PASS: "[PASS]",
        GuardrailVerdict.WARN: "[WARN]",
        GuardrailVerdict.BLOCK: "[BLOCK]",
    }[scenario["verdict"]]
    
    print(f"\n  Scenario {i}: {verdict_symbol}")
    print(f"    Message: '{scenario['message'][:70]}...'")
    print(f"    Content safe: {scenario['content_safe']}")
    print(f"    PII detected: {scenario['pii_detected']}")
    if scenario["warnings"]:
        print(f"    Warnings: {scenario['warnings']}")
    if scenario.get("blocked_reason"):
        print(f"    Blocked reason: {scenario['blocked_reason']}")

In [ ]:
# Demonstrate groundedness checking on output
print("=== Output Groundedness Verification ===")

groundedness_scenarios = [
    {
        "label": "Well-grounded response",
        "context": "Art. 7:672 BW: opzegtermijn werkgever 5-10 jaar dienstverband: twee maanden.",
        "response": "De opzegtermijn bedraagt twee maanden conform art. 7:672 BW.",
        "result": GroundednessResult(
            score=0.95,
            is_grounded=True,
            ungrounded_claims=[],
            explanation="All claims are directly supported by the source context.",
        ),
    },
    {
        "label": "Partially grounded (hallucinated reference)",
        "context": "Art. 7:672 BW: opzegtermijn werkgever 5-10 jaar dienstverband: twee maanden.",
        "response": (
            "De opzegtermijn bedraagt twee maanden conform art. 7:672 BW. "
            "Bovendien heeft de HR in ECLI:NL:HR:2023:9999 bevestigd dat bij "
            "reorganisaties een extra maand geldt."
        ),
        "result": GroundednessResult(
            score=0.45,
            is_grounded=False,
            ungrounded_claims=[
                "ECLI:NL:HR:2023:9999 is not present in the source context",
                "Claim about extra month for reorganizations is not supported",
            ],
            explanation="The response contains fabricated case law reference and unsupported claim.",
        ),
    },
    {
        "label": "Ungrounded (wrong information)",
        "context": "Art. 7:672 BW: opzegtermijn werkgever 5-10 jaar dienstverband: twee maanden.",
        "response": "De opzegtermijn bedraagt vier maanden op grond van artikel 7:672a BW.",
        "result": GroundednessResult(
            score=0.15,
            is_grounded=False,
            ungrounded_claims=[
                "'vier maanden' contradicts source which states 'twee maanden'",
                "'artikel 7:672a BW' does not exist in the source context",
            ],
            explanation="Response contains factual errors and a non-existent article reference.",
        ),
    },
]

for scenario in groundedness_scenarios:
    result = scenario["result"]
    status = "PASS" if result.is_grounded else "FAIL"
    
    print(f"\n--- {scenario['label']} ---")
    print(f"  Response: '{scenario['response'][:80]}...'")
    print(f"  Score: {result.score:.2f} (threshold: 0.60)")
    print(f"  Verdict: {status}")
    if result.ungrounded_claims:
        print(f"  Ungrounded claims:")
        for claim in result.ungrounded_claims:
            print(f"    - {claim}")
    print(f"  Explanation: {result.explanation}")

## 7. Performance Summary

Let's summarize the key performance characteristics of the multi-agent system.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated latency breakdown for different request types
request_types = [
    "Simple Q&A",
    "Document\nAnalysis",
    "Legal\nResearch",
    "Drafting",
    "Research +\nDrafting\n(handoff)",
    "Compliance\nAudit",
]

# Latency components (in seconds)
input_guardrails = [0.3, 0.3, 0.3, 0.3, 0.3, 0.3]
intent_classification = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
tool_execution = [0.0, 0.8, 1.2, 0.4, 1.6, 1.0]
llm_generation = [1.5, 2.5, 3.0, 2.8, 5.5, 3.5]
output_guardrails = [0.4, 0.5, 0.5, 0.4, 0.6, 0.5]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Stacked bar chart for latency breakdown
ax = axes[0]
x = np.arange(len(request_types))
width = 0.6

components = [
    (input_guardrails, "Input Guardrails", "#3498db"),
    (intent_classification, "Intent Classification", "#e74c3c"),
    (tool_execution, "Tool Execution", "#f39c12"),
    (llm_generation, "LLM Generation", "#2ecc71"),
    (output_guardrails, "Output Guardrails", "#9b59b6"),
]

bottom = np.zeros(len(request_types))
for values, label, color in components:
    ax.bar(x, values, width, bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += np.array(values)

# Add total time labels on top
totals = [sum(v[i] for v, _, _ in components) for i in range(len(request_types))]
for i, total in enumerate(totals):
    ax.text(i, total + 0.1, f"{total:.1f}s", ha='center', fontweight='bold', fontsize=10)

ax.set_xlabel('Request Type', fontsize=12)
ax.set_ylabel('Latency (seconds)', fontsize=12)
ax.set_title('End-to-End Latency Breakdown', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(request_types, fontsize=9)
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, 10)
ax.grid(axis='y', alpha=0.3)

# Right: Agent usage distribution (simulated from 1000 production queries)
ax = axes[1]
agent_usage = {
    "Document Analysis": 285,
    "Legal Research": 310,
    "Drafting": 195,
    "Compliance": 110,
    "General": 100,
}

agent_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#95a5a6"]
wedges, texts, autotexts = ax.pie(
    agent_usage.values(),
    labels=agent_usage.keys(),
    autopct='%1.1f%%',
    colors=agent_colors,
    startangle=90,
    pctdistance=0.75,
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
ax.set_title('Agent Usage Distribution\n(simulated: 1000 queries)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("=" * 70)
print("LEGALMIND MULTI-AGENT SYSTEM -- PERFORMANCE SUMMARY")
print("=" * 70)

summary = [
    ("Architecture", "Multi-agent with Semantic Kernel + GPT-4o"),
    ("Specialist agents", "4 (Document, Research, Drafting, Compliance)"),
    ("Total tools (SK plugins)", "5 plugins, 14 functions"),
    ("Intent classification", "GPT-4o @ temp=0.1 (JSON output)"),
    ("Agent handoffs", "Enabled (max depth: 3)"),
    ("", ""),
    ("Avg latency (simple)", "2.7 seconds"),
    ("Avg latency (with tools)", "4.5 seconds"),
    ("Avg latency (handoff)", "8.4 seconds"),
    ("", ""),
    ("Input guardrails", "Content safety + PII detection"),
    ("Output guardrails", "Content safety + PII + Groundedness"),
    ("Groundedness threshold", "0.6 (configurable)"),
    ("Groundedness eval temp", "0.1 (strict)"),
    ("", ""),
    ("Authentication", "Azure AD (DefaultAzureCredential)"),
    ("Conversation history", "50 messages per agent (trimmed)"),
    ("Max tokens per response", "4096"),
]

for label, value in summary:
    if label:
        print(f"  {label:<30} {value}")
    else:
        print()

## Summary

This notebook demonstrated the full LegalMind-AI multi-agent architecture:

1. **Intent Classification**: The Orchestrator uses GPT-4o with a specialized prompt to
   route user messages to the correct specialist agent with high accuracy (avg confidence: 0.87).

2. **Specialist Agents**: Four domain-specific agents (Document, Research, Drafting, Compliance)
   each with tailored system prompts, few-shot examples, and tool access via Semantic Kernel.

3. **Agent Handoffs**: When a request spans multiple domains (e.g., research + drafting),
   agents can delegate to each other while passing analysis context.

4. **Tool Calling**: Agents automatically invoke tools (search, extract, summarize, draft,
   translate) as needed, with GPT-4o deciding the tool calling strategy.

5. **Responsible AI Guardrails**: Three-layer protection (input, processing, output) with
   content safety, PII detection, and groundedness verification ensures safe, accurate
   responses in the sensitive legal domain.

The architecture is designed for the Dutch legal domain at Voorbeeld Rechtsbijstand, handling
contracts, case law, correspondence, and compliance checks in both Dutch and English.